In [1]:
!pip install -q librosa timm

import os, glob, random
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

cuda


In [2]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
g2i = {g: i for i, g in enumerate(GENRES)}
STEMS = ['drums', 'vocals', 'bass', 'other']
BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'

SR, DUR, N_MELS = 22050, 10, 128

songs = {g: [] for g in GENRES}
for g in GENRES:
    gd = os.path.join(BASE, 'genres_stems', g)
    if os.path.exists(gd):
        for s in os.listdir(gd):
            sp = os.path.join(gd, s)
            if os.path.isdir(sp) and all(os.path.exists(os.path.join(sp, f"{st}.wav")) for st in STEMS):
                songs[g].append(sp)

noise = glob.glob(os.path.join(BASE, 'ESC-50-master', 'audio', '*.wav'))
print(f"Songs: {sum(len(v) for v in songs.values())}, Noise: {len(noise)}")

Songs: 1000, Noise: 2000


In [3]:
def load(p, sr=SR, d=DUR):
    try:
        a, _ = librosa.load(p, sr=sr, duration=d)
        t = sr * d
        if len(a) < t: a = np.tile(a, 3)[:t]
        return a[:t]
    except: return np.zeros(sr * d)

def mix_cross(g):
    m = np.zeros(SR * DUR, dtype=np.float32)
    for st in STEMS:
        m += load(os.path.join(random.choice(songs[g]), f"{st}.wav"))
    return m

def add_noise(a, lvl):
    n = load(random.choice(noise)) if noise else np.zeros_like(a)
    if np.max(np.abs(n)) > 0: n /= np.max(np.abs(n))
    return a + lvl * n

def norm(a):
    a = a - np.mean(a)
    if np.max(np.abs(a)) > 0: a = a / np.max(np.abs(a)) * 0.95
    return a.astype(np.float32)

def to_mel(a):
    m = librosa.feature.melspectrogram(y=a, sr=SR, n_mels=N_MELS, n_fft=2048, hop_length=512)
    m = librosa.power_to_db(m, ref=np.max)
    return (m - m.mean()) / (m.std() + 1e-6)

In [4]:
class DS(Dataset):
    def __init__(self, n=200):
        self.d = [(g, i) for g in GENRES for i in range(n)]
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        g, _ = self.d[i]
        a = mix_cross(g)
        if random.random() < 0.75:
            a = add_noise(a, random.uniform(0.1, 0.35))
        if random.random() < 0.5:
            a = np.roll(a, random.randint(-SR//2, SR//2))
        a = norm(a)
        m = to_mel(a)
        if random.random() < 0.5:
            t = random.randint(0, 25)
            t0 = random.randint(0, max(1, m.shape[1]-t-1))
            m[:, t0:t0+t] = 0
        if random.random() < 0.5:
            f = random.randint(0, 15)
            f0 = random.randint(0, max(1, m.shape[0]-f-1))
            m[f0:f0+f, :] = 0
        return torch.tensor(m, dtype=torch.float32).unsqueeze(0).repeat(3,1,1), g2i[g]

class Model(nn.Module):
    def __init__(self, arch='efficientnet_b2'):
        super().__init__()
        self.arch = arch
        self.bb = timm.create_model(arch, pretrained=True, num_classes=0)
        self.h = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(self.bb.num_features, 256),
            nn.ReLU(), nn.Dropout(0.25), nn.Linear(256, 10)
        )
    def forward(self, x): return self.h(self.bb(x))

In [5]:
def train_model(arch, seed, epochs=12):
    print(f"\n{'='*40}")
    print(f"TRAINING {arch} SEED {seed}")
    print(f"{'='*40}")
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    model = Model(arch).to(device)
    tr = DataLoader(DS(200), batch_size=24, shuffle=True, num_workers=2)
    
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=8e-4, epochs=epochs, steps_per_epoch=len(tr))
    
    best = 0
    for ep in range(epochs):
        model.train()
        c, t = 0, 0
        for d, l in tqdm(tr, desc=f"{arch[:6]}-{seed} Ep{ep+1}"):
            d, l = d.to(device), l.to(device)
            opt.zero_grad()
            o = model(d)
            crit(o, l).backward()
            opt.step()
            sch.step()
            c += (o.argmax(1) == l).sum().item()
            t += l.size(0)
        acc = c / t
        if acc > best:
            best = acc
            torch.save(model.state_dict(), f'model_{arch}_{seed}.pth')
        print(f"Acc: {acc:.4f}, Best: {best:.4f}")
    
    model.load_state_dict(torch.load(f'model_{arch}_{seed}.pth'))
    return model

In [6]:
# DIVERSE ENSEMBLE: Different architectures!
# 2x EfficientNet-B2, 1x EfficientNet-B3, 1x ResNet34
configs = [
    ('efficientnet_b2', 42),
    ('efficientnet_b2', 123),
    ('efficientnet_b3', 42),
    ('resnet34', 42),
]

models = []
for arch, seed in configs:
    m = train_model(arch, seed, epochs=12)
    models.append((m, arch, seed))
print(f"\nTrained {len(models)} diverse models!")


TRAINING efficientnet_b2 SEED 42


model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

effici-42 Ep1: 100%|██████████| 84/84 [04:29<00:00,  3.21s/it]


Acc: 0.2450, Best: 0.2450


effici-42 Ep2: 100%|██████████| 84/84 [03:37<00:00,  2.59s/it]


...

Acc: 0.8695, Best: 0.8695


resnet-42 Ep11: 100%|██████████| 84/84 [03:29<00:00,  2.49s/it]


Acc: 0.8765, Best: 0.8765


resnet-42 Ep12: 100%|██████████| 84/84 [03:32<00:00,  2.53s/it]


Acc: 0.8825, Best: 0.8825

Trained 4 diverse models!


In [7]:
# Load and eval
for i, (model, arch, seed) in enumerate(models):
    model.load_state_dict(torch.load(f'model_{arch}_{seed}.pth'))
    model.eval()

test_df = pd.read_csv(os.path.join(BASE, 'test.csv'))
sub = pd.read_csv(os.path.join(BASE, 'sample_submission.csv'))
i2f = dict(zip(test_df['id'], test_df['filename']))
files = [os.path.join(BASE, i2f[r['id']]) for _, r in sub.iterrows()]
print(f"Test files: {len(files)}")

Test files: 3020


In [8]:
def ensemble_tta(path, n_crops=5):
    try:
        af, _ = librosa.load(path, sr=SR)
    except:
        af = np.zeros(SR * 20)
    
    t = SR * DUR
    if len(af) < t: af = np.tile(af, 3)
    
    all_probs = []
    
    for model, arch, seed in models:
        for p in np.linspace(0, max(0, len(af)-t), n_crops).astype(int):
            cr = af[p:p+t]
            if len(cr) < t: cr = np.pad(cr, (0, t-len(cr)))
            cr = norm(cr)
            m = to_mel(cr)
            mt = torch.tensor(m, dtype=torch.float32).unsqueeze(0).unsqueeze(0).repeat(1,3,1,1).to(device)
            with torch.no_grad():
                all_probs.append(torch.softmax(model(mt), dim=1))
    
    # 4 models x 5 crops = 20 predictions averaged
    return torch.stack(all_probs).mean(0).argmax(1).item()

print(f"Diverse Ensemble (2xB2 + 1xB3 + 1xResNet34) x 5 crops...")
preds = [ensemble_tta(f) for f in tqdm(files)]

sub['genre'] = [GENRES[p] for p in preds]
sub.to_csv('submission.csv', index=False)
print("Done!")
print(sub['genre'].value_counts())

Diverse Ensemble (2xB2 + 1xB3 + 1xResNet34) x 5 crops...


100%|██████████| 3020/3020 [38:01<00:00,  1.32it/s]

Done!
genre
pop          327
reggae       323
rock         320
classical    313
disco        313
metal        302
hiphop       300
jazz         290
blues        273
country      259
Name: count, dtype: int64
